In [0]:
%sql
CREATE TABLE IF NOT EXISTS voyage.monitoring.pipeline_run_audit (
  run_id            STRING        COMMENT 'Unique run identifier (Airflow run_id or UUID)',
  pipeline_name     STRING        COMMENT 'Logical pipeline name, e.g. voyage_daily_pipeline',
  started_at        TIMESTAMP     COMMENT 'When the pipeline run started',
  ended_at          TIMESTAMP     COMMENT 'When the pipeline run ended (null if still running)',
  status            STRING        COMMENT 'Overall status: RUNNING, SUCCESS, FAILED',
  airbyte_status    STRING        COMMENT 'Airbyte sync status: PENDING, SUCCESS, FAILED, SKIPPED',
  databricks_status STRING        COMMENT 'Databricks notebook job status: PENDING, SUCCESS, FAILED, SKIPPED',
  dbt_status        STRING        COMMENT 'dbt model build status: PENDING, SUCCESS, FAILED, SKIPPED',
  dq_status         STRING        COMMENT 'Data quality check status: PENDING, PASS, FAIL, SKIPPED',
  freshness_status  STRING        COMMENT 'Data freshness SLA status: PENDING, PASS, FAIL, SKIPPED',
  records_processed BIGINT        COMMENT 'Total records processed in this run',
  error_message     STRING        COMMENT 'First error message if any stage failed (null on success)'
)
USING DELTA
COMMENT 'Audit log — one row per pipeline run with per-stage status'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
%sql
DESCRIBE TABLE voyage.monitoring.pipeline_run_audit;

## 11.3 — Data Quality Monitoring Table

Stores the result of every data quality check per pipeline run. One row per check per run.

**Grain**: `pipeline_run_id` x `check_name` = one row

| Column | Purpose |
|---|---|
| `pipeline_run_id` | Links to `pipeline_run_audit.run_id` |
| `check_name` | e.g. `booking_duplicate_check` |
| `domain` | e.g. `bookings`, `marketing`, `website`, `attribution` |
| `check_type` | `completeness`, `uniqueness`, `referential`, `validity`, `reconciliation`, `business_rule` |
| `status` | `PASS` or `FAIL` |
| `expected_value` | What we expect (e.g. `0` for duplicate count) |
| `actual_value` | What we found (e.g. `0` or `3`) |
| `checked_at` | Timestamp of the check |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS voyage.monitoring.data_quality_results (
  pipeline_run_id   STRING        NOT NULL COMMENT 'Links to pipeline_run_audit.run_id',
  check_name        STRING        NOT NULL COMMENT 'e.g. booking_duplicate_check',
  domain            STRING        NOT NULL COMMENT 'e.g. bookings, marketing, website, attribution',
  check_type        STRING        NOT NULL COMMENT 'completeness, uniqueness, referential, validity, reconciliation, business_rule',
  status            STRING        NOT NULL COMMENT 'PASS or FAIL',
  expected_value    STRING        COMMENT 'What we expect (e.g. 0 for duplicates)',
  actual_value      STRING        COMMENT 'What we found (e.g. 0 or 3)',
  checked_at        TIMESTAMP     NOT NULL COMMENT 'When the check ran'
)
USING DELTA
COMMENT 'DQ check results — one row per check per pipeline run'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
%sql
DESCRIBE TABLE voyage.monitoring.data_quality_results;

## 11.4 — Freshness Monitoring

Before creating the freshness table, we inspect available timestamp columns to find the safest freshness timestamp per domain.

**Plan**: Prefer `_airbyte_extracted_at` (ingestion timestamp) over business event dates.

**SLA thresholds** (portfolio-level):
- < 2 hours → `PASS`
- 2–6 hours → `WARN`
- > 6 hours → `ERROR`

In [0]:
%sql
-- Inspect available timestamp columns in bronze tables
SHOW TABLES IN voyage.bronze;

In [0]:
%sql
-- Check latest _airbyte_extracted_at per domain (key bronze tables)
SELECT 'bookings'           AS dataset, max(to_timestamp(_airbyte_extracted_at)) AS max_loaded_at FROM voyage.bronze.bookings
UNION ALL
SELECT 'marketing_bookings',  max(to_timestamp(_airbyte_extracted_at)) FROM voyage.bronze.marketing_bookings
UNION ALL
SELECT 'marketing_campaigns', max(to_timestamp(_airbyte_extracted_at)) FROM voyage.bronze.marketing_campaigns
UNION ALL
SELECT 'marketing_campaign_performance', max(to_timestamp(_airbyte_extracted_at)) FROM voyage.bronze.marketing_campaign_performance
UNION ALL
SELECT 'website_events',      max(to_timestamp(_airbyte_extracted_at)) FROM voyage.bronze.website_events
UNION ALL
SELECT 'website_sessions',    max(to_timestamp(_airbyte_extracted_at)) FROM voyage.bronze.website_sessions
UNION ALL
SELECT 'website_page_views',  max(to_timestamp(_airbyte_extracted_at)) FROM voyage.bronze.website_page_views
ORDER BY max_loaded_at DESC;

In [0]:
%sql
-- Check latest _airbyte_extracted_at per domain using timestamp_millis (epoch ms)
SELECT 'bookings'           AS dataset, max(timestamp_millis(_airbyte_extracted_at)) AS max_loaded_at FROM voyage.bronze.bookings
UNION ALL
SELECT 'marketing_bookings',  max(timestamp_millis(_airbyte_extracted_at)) FROM voyage.bronze.marketing_bookings
UNION ALL
SELECT 'marketing_campaigns', max(timestamp_millis(_airbyte_extracted_at)) FROM voyage.bronze.marketing_campaigns
UNION ALL
SELECT 'marketing_campaign_performance', max(timestamp_millis(_airbyte_extracted_at)) FROM voyage.bronze.marketing_campaign_performance
UNION ALL
SELECT 'website_events',      max(timestamp_millis(_airbyte_extracted_at)) FROM voyage.bronze.website_events
UNION ALL
SELECT 'website_sessions',    max(timestamp_millis(_airbyte_extracted_at)) FROM voyage.bronze.website_sessions
UNION ALL
SELECT 'website_page_views',  max(timestamp_millis(_airbyte_extracted_at)) FROM voyage.bronze.website_page_views
ORDER BY max_loaded_at DESC;

In [0]:
%sql
-- List silver tables to see what's available
SHOW TABLES IN voyage.silver;

### Freshness Timestamp Findings

**Format**: `_airbyte_extracted_at` is stored as **epoch milliseconds** (LONG) in bronze tables.
Must use `timestamp_millis()` — NOT `to_timestamp()` (which treats values as seconds and gives year 58719).

**Freshness source**: Bronze layer (where `_airbyte_extracted_at` is available). Silver drops this column.

| Domain | Bronze Table(s) | Freshness Column |
|---|---|---|
| Bookings | `voyage.bronze.bookings` | `timestamp_millis(_airbyte_extracted_at)` |
| Marketing Bookings | `voyage.bronze.marketing_bookings` | same |
| Marketing Campaigns | `voyage.bronze.marketing_campaigns` | same |
| Marketing Campaign Perf | `voyage.bronze.marketing_campaign_performance` | same |
| Website Events | `voyage.bronze.website_events` | same |
| Website Sessions | `voyage.bronze.website_sessions` | same |
| Website Page Views | `voyage.bronze.website_page_views` | same |

**SLA Thresholds** (portfolio-level):
- < 2 hours -> `PASS`
- 2-6 hours -> `WARN`
- > 6 hours -> `ERROR`

In [0]:
%sql
CREATE TABLE IF NOT EXISTS voyage.monitoring.freshness_results (
  pipeline_run_id   STRING        NOT NULL COMMENT 'Links to pipeline_run_audit.run_id',
  dataset           STRING        NOT NULL COMMENT 'e.g. bookings, marketing_campaigns, website_events',
  max_loaded_at     TIMESTAMP     NOT NULL COMMENT 'Latest ingestion timestamp from bronze _airbyte_extracted_at',
  checked_at        TIMESTAMP     NOT NULL COMMENT 'When the freshness check ran',
  age_minutes       BIGINT        NOT NULL COMMENT 'Minutes between max_loaded_at and checked_at',
  status            STRING        NOT NULL COMMENT 'PASS (<2h), WARN (2-6h), ERROR (>6h)'
)
USING DELTA
COMMENT 'Freshness check results — one row per dataset per pipeline run'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
%sql
DESCRIBE TABLE voyage.monitoring.freshness_results;

In [0]:
%sql
-- Verify all three monitoring tables exist
SHOW TABLES IN voyage.monitoring;

In [0]:
%sql
-- Full audit: list ALL bronze + silver + dbt_dev tables
SHOW TABLES IN voyage.bronze;

In [0]:
%sql
-- Check freshness results written by the validation notebook
SELECT dataset, max_loaded_at, checked_at, age_minutes, status
FROM voyage.monitoring.freshness_results
ORDER BY age_minutes ASC;